# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aleezafatima-21/Aleeza-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one content item for one client on one reporting day.

Time window: I will use March 2026 as the development and verification month. I will treat June 2026 as a sealed final test month and will not use the _sample table for label development.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", bool(os.environ["HF_TOKEN"]))

HF_TOKEN loaded: True


In [2]:
!pip -q install duckdb huggingface_hub

In [7]:
import duckdb

con = duckdb.connect()

print("DuckDB connected:", con is not None)

DuckDB connected: True


In [8]:
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

print("Hugging Face login successful")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful


In [9]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem(token=os.environ["HF_TOKEN"])

files = fs.glob("datasets/FlyRank/internship-warehouse/*")

print("Warehouse access successful.")
print("Files found:", len(files))
print(files[:5])

Warehouse access successful.
Files found: 7
['datasets/FlyRank/internship-warehouse/.gitattributes', 'datasets/FlyRank/internship-warehouse/README.md', 'datasets/FlyRank/internship-warehouse/dim_clients.parquet', 'datasets/FlyRank/internship-warehouse/dim_content.parquet', 'datasets/FlyRank/internship-warehouse/fact_content_daily_performance']


In [10]:
files = fs.glob(
    "datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*"
)

print("Files found:", len(files))
print("\n".join(files[:10]))

Files found: 18
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-09
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-10


In [12]:
import os
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(
    "CREATE OR REPLACE SECRET hf_secret "
    "(TYPE HUGGINGFACE, TOKEN ?)",
    [os.environ["HF_TOKEN"]]
)

print("DuckDB Hugging Face authentication configured.")

DuckDB Hugging Face authentication configured.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:
- `gsc_impressions` — historical search visibility available before the decision moment.
- `gsc_clicks` — historical search traffic available before the decision moment.
- `gsc_avg_position` — historical search ranking information available before the decision moment.
- `ga4_sessions` — historical website sessions available before the decision moment when GA4 is available.
- `ga4_engaged_sessions` — historical engaged sessions available before the decision moment when GA4 is available.

Label / proxy:
- A future performance-decline outcome used to rank content items for refresh.

Context:
- `client_hash_id`, `content_hash_id`, `report_date` — used to identify content, group data, and define the time window, not as model features.

Excluded:
- `trend_pct` and `trend_direction` — excluded because they are used to derive the decline label and would leak the outcome into the features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain check: March 2026 has no duplicate client-content-date combinations, so the stated daily grain holds.

Query 1 — **Grain**

In [16]:
query = """
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.execute(query).df()

print("Duplicate grain combinations found:", len(grain_check))
display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations found: 0


,client_hash_id,content_hash_id,report_date,row_count


Query 2 — Counts + date **span**

In [17]:
query = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

count_window_check = con.execute(query).df()

display(count_window_check)

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


Query 3 — **Availability**

Availability check: 413,966 March 2026 rows have confirmed GA4 availability using ga4_data_available IS TRUE.

In [18]:
query = """
SELECT
    COUNT(*) AS rows_with_ga4_available
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
"""

availability_check = con.execute(query).df()

display(availability_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_ga4_available
0,413966


In [19]:
columns = con.execute("""
    DESCRIBE SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

display(columns)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [20]:
label_columns = columns[
    columns["column_name"].str.contains(
        "trend|declin|label|change|prev|next",
        case=False,
        regex=True
    )
]

display(label_columns)

,column_name,column_type,null,key,default,extra


In [21]:
content_columns = con.execute("""
    DESCRIBE SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
""").df()

display(content_columns)

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [22]:
feature_frame = con.execute("""
    SELECT
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    GROUP BY content_hash_id
""").df()

display(feature_frame.head())
print("Feature frame rows:", len(feature_frame))
print("Feature frame columns:", list(feature_frame.columns))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,0.0
1,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0
2,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,0.0
3,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,0.0
4,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,0.0


Feature frame rows: 331437
Feature frame columns: ['content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']


### Why each feature is available at the decision moment

- `gsc_impressions` — available at the decision moment because it is measured from the March performance window.
- `gsc_clicks` — available at the decision moment because it is measured from the March performance window.
- `gsc_avg_position` — available at the decision moment because it is calculated from March search-performance observations.
- `ga4_sessions` — available at the decision moment because it comes from the March performance window and is used only when `ga4_data_available IS TRUE`.
- `ga4_engaged_sessions` — available at the decision moment because it comes from the March performance window and is used only when `ga4_data_available IS TRUE`.

In [23]:
coverage_check = con.execute("""
    SELECT
        month,
        COUNT(DISTINCT content_hash_id) AS content_items,
        COUNT(*) AS daily_rows,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM read_parquet([
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    ])
    GROUP BY month
    ORDER BY month
""").df()

display(coverage_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,content_items,daily_rows,start_date,end_date
0,2026-03,331437,9841378,2026-03-01,2026-03-31
1,2026-04,362172,10424730,2026-04-01,2026-04-30


In [24]:
label_data = con.execute("""
    WITH march AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS march_impressions
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        )
        GROUP BY content_hash_id
    ),

    april AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
        )
        GROUP BY content_hash_id
    )

    SELECT
        march.content_hash_id,
        march.march_impressions,
        april.april_impressions,

        CASE
            WHEN april.april_impressions < march.march_impressions
            THEN 1
            ELSE 0
        END AS is_declining_label

    FROM march
    INNER JOIN april
        ON march.content_hash_id = april.content_hash_id
""").df()

display(label_data.head())

print("Rows with both March and April data:", len(label_data))
print("Declining:", label_data["is_declining_label"].sum())
print("Not declining:", (label_data["is_declining_label"] == 0).sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,march_impressions,april_impressions,is_declining_label
0,content_ddbfb1907979759a,13.0,8.0,1
1,content_19d31b32f74b4f12,6.0,6.0,0
2,content_92bc8dfb830d0ade,13.0,26.0,0
3,content_2a44e78f3d53769e,2.0,17.0,0
4,content_745efcdf75e0ec8c,13.0,7.0,1


Rows with both March and April data: 331436
Declining: 111967
Not declining: 219469


In [25]:
# Deliberate leakage experiment:
# April impressions are future information and should NOT be a feature.

leaky_frame = feature_frame.merge(
    label_data[["content_hash_id", "april_impressions", "is_declining_label"]],
    on="content_hash_id",
    how="inner"
)

print("Rows in leaky frame:", len(leaky_frame))

leak_score = leaky_frame.groupby("is_declining_label")["april_impressions"].mean()

print("\nAverage April impressions by label:")
print(leak_score)

Rows in leaky frame: 331436

Average April impressions by label:
is_declining_label
0     775.702846
1    1033.895576
Name: april_impressions, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Label/proxy: I will rank content items by their future performance decline/opportunity for refresh, using a future outcome as the label/proxy.

### Data limits

This dataset supports directional decision-support, but it cannot provide a perfectly balanced view of every client or content item.

- Client history is unbalanced. Some clients have much deeper search and analytics history than others, so observations are not equally comparable across clients.
- Early rows can be GSC-only. GA4 fields may be zero-filled before `ga4_data_start`, so those zeros cannot be interpreted as genuine zero engagement. `ga4_data_available` must be checked.
- The label window overlaps with some available performance fields. For example, April impressions are part of the outcome period used to determine whether a content item is declining. They are therefore future information and cannot be used as prediction features.
- The March-to-April check produced 331,436 content items with data in both months. The observed label distribution was 111,967 declining and 219,469 not declining. This is an imbalanced outcome, so model performance should not be judged by accuracy alone.
- The data can identify patterns associated with declining content, but it cannot prove that a particular factor caused the decline.
- The analysis is therefore intended to rank content for review and provide decision support, not to guarantee which pages will decline or explain causality.

The warehouse has uneven history across clients, so the available observation window is not equally reliable for every client. Some early rows are GSC-only because GA4 was not yet available; these rows should not be interpreted as zero engagement.

The March 2026 development window contains 9,841,378 daily rows, covering 2026-03-01 to 2026-03-31. I found 331,436 content items with both March and April data, which is the population used for this development check.

The March → April label is useful for testing label construction, but it also shows a major leakage risk: April is the outcome window, so April metrics must never be included as features when predicting the April outcome.

The label distribution in this development frame is 66.2% not declining and 33.8% declining. Therefore, the dataset is not perfectly balanced and accuracy alone should not be treated as sufficient evidence of model quality.

The warehouse also contains overlapping 90-day query windows. Query-table context fields must therefore be aligned carefully with the snapshot date; fields that overlap the future outcome window must not be used as features.

This data can support directional, decision-support predictions about content performance. It cannot establish that a particular content change caused a performance change, and it cannot guarantee future traffic or ranking outcomes for every client.

The warehouse has several limits that affect what the model can tell us.

- **History is unbalanced across clients.** Clients have different GSC and GA4 starting dates, so the same calendar window does not represent the same amount of history for every client. Results should therefore be interpreted as decision-support rather than a complete historical view.

- **Some early rows are GSC-only.** Before a client's GA4 data becomes available, GA4 fields may be zero-filled while `ga4_data_available = FALSE`. These zeros cannot be interpreted as zero engagement.

- **The final month must remain sealed.** The June 2026 month is the natural final outcome window, so it should not be used while developing the label or features.

- **Window overlap can cause leakage.** The query table contains a fixed 90-day window that can overlap the outcome period. Features must be aligned so that they only use information available before the prediction point.

- **The data does not establish causation.** A relationship between search performance, content characteristics, and decline does not prove that one factor caused another.

- **The model cannot tell us what will happen for every client or content item.** Clients with limited usable history may need to be filtered, and predictions should be treated as directional decision-support rather than certainty.

In [27]:
# Verify the main data limitations used in the contract

print("Rows with both March and April data:", len(label_data))

print("\nLabel distribution:")
print(label_data["is_declining_label"].value_counts())

print("\nLabel proportions:")
print(label_data["is_declining_label"].value_counts(normalize=True))

print("\nRows in leaky frame:", len(leaky_frame))

print("\nAverage April impressions by label:")
print(
    leaky_frame.groupby("is_declining_label")["april_impressions"].mean()
)

Rows with both March and April data: 331436

Label distribution:
is_declining_label
0    219469
1    111967
Name: count, dtype: int64

Label proportions:
is_declining_label
0    0.662176
1    0.337824
Name: proportion, dtype: float64

Rows in leaky frame: 331436

Average April impressions by label:
is_declining_label
0     775.702846
1    1033.895576
Name: april_impressions, dtype: float64


In [28]:
# Verify the main data limitations used in the contract

# 1. Check GA4 availability in the March development month
ga4_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN NOT ga4_data_available THEN 1 ELSE 0 END) AS ga4_unavailable_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

print("GA4 availability in March 2026:")
display(ga4_check)


# 2. Check the development and final months separately
month_check = con.execute("""
    SELECT
        '2026-03' AS month,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(*) AS rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    UNION ALL

    SELECT
        '2026-06' AS month,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(*) AS rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/*.parquet'
    )
""").df()

print("\nDevelopment vs final month:")
display(month_check)


# 3. Check whether GA4-unavailable rows contain zero-filled GA4 values
zero_fill_check = con.execute("""
    SELECT
        COUNT(*) AS ga4_unavailable_rows,
        SUM(
            CASE
                WHEN ga4_pageviews = 0
                 AND ga4_sessions = 0
                 AND ga4_users = 0
                 AND ga4_engaged_sessions = 0
                THEN 1 ELSE 0
            END
        ) AS rows_with_zero_ga4_metrics
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE ga4_data_available = FALSE
""").df()

print("\nGA4 unavailable / zero-filled check:")
display(zero_fill_check)

GA4 availability in March 2026:


,total_rows,ga4_available_rows,ga4_unavailable_rows
0,9841378,413966.0,6408671.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Development vs final month:


,month,start_date,end_date,rows
0,2026-03,2026-03-01,2026-03-31,9841378
1,2026-06,2026-06-01,2026-06-30,11694072



GA4 unavailable / zero-filled check:


,ga4_unavailable_rows,rows_with_zero_ga4_metrics
0,6408671,6408671.0


### Observed checks

In March 2026, the warehouse contains 9,841,378 daily rows. Only 413,966 rows have GA4 available, while 6,408,671 rows have `ga4_data_available = FALSE`. All of those unavailable rows have zero-valued GA4 metrics, confirming that these zeros should not be interpreted as zero engagement.

March 2026 covers 2026-03-01 to 2026-03-31 and was used as the development month. June 2026 covers 2026-06-01 to 2026-06-30 and contains 11,694,072 rows, so it should remain sealed as the final test/outcome month.

These checks support treating the data as directional decision-support rather than a complete or perfectly balanced historical record.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.